# 04 — Main runner

Executes the pipeline notebooks in order (replaces the old `run_pipeline.sh`):

| step | notebook | needs | produces |
|---|---|---|---|
| 00 | `00_setup` | ecoinvent, CF layers | Brightway project + methods |
| 01 | `01_scenarios` | 00 | all deterministic tables |
| 02 | `02_monte_carlo` | 01 | `mc_*` tables (long; optional) |
| 03 | `03_results` | 01 (02 for MC figures) | every figure |

Failures upstream abort downstream steps. Log: `results/pipeline_run.log`.


In [ ]:
# ── Flags ────────────────────────────────────────────────────────
RUN_SETUP     = True    # 00_setup: only needed after inventory/CF changes
RUN_SCENARIOS = True    # 01_scenarios: rebuilds all tables
RUN_MC        = False   # 02_monte_carlo: overnight job; figures skip
                        # politely without it
RUN_RESULTS   = True    # 03_results: all figures


In [ ]:
import os, subprocess, sys, time

LOG = os.path.join("results", "pipeline_run.log")
os.makedirs("results", exist_ok=True)

STEPS = [("00_setup.ipynb",       RUN_SETUP,     True),
         ("01_scenarios.ipynb",   RUN_SCENARIOS, True),
         ("02_monte_carlo.ipynb", RUN_MC,        False),
         ("03_results.ipynb",     RUN_RESULTS,   False)]
# third field: abort the remaining steps if this one fails

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")

log("PIPELINE START")
aborted = False
for nb_name, enabled, critical in STEPS:
    if aborted:
        log(f"SKIP  {nb_name} (upstream failure)")
        continue
    if not enabled:
        log(f"SKIP  {nb_name} (flag off)")
        continue
    log(f"START {nb_name}")
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook",
         "--execute", "--inplace",
         "--ExecutePreprocessor.timeout=-1", nb_name],
        capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode == 0:
        log(f"OK    {nb_name}  ({dt/60:.1f} min)")
    else:
        log(f"FAIL  {nb_name}  ({dt/60:.1f} min)  rc={r.returncode}")
        with open(LOG, "a", encoding="utf-8") as fh:
            fh.write(r.stderr[-4000:] + "\n")
        print(r.stderr[-2000:])
        if critical:
            aborted = True
log("PIPELINE END" + (" (ABORTED)" if aborted else ""))
